In [ ]:
import pathway as pw
from pathway.xpacks.llm.splitters import TokenCountSplitter
import re

# 1. Define the Schema for Ingested Data
class NovelSchema(pw.Schema):
    data: str # The content of the novel file

def get_preprocessing_pipeline(input_dir: str):
    # 2. Ingestion: Read local .txt novels using 'plaintext_by_file'
    # This treats each novel as one row, preserving its global sequence.
    raw_novels = pw.io.fs.read(
        input_dir,
        format="plaintext_by_file",
        mode="static", # Use 'streaming' for real-time monitoring
        with_metadata=True
    )

    # 3. Preprocessing: Structural (Hierarchical) Chunking
    # We first split by common chapter markers to preserve narrative boundaries.
    def split_into_chapters(text: str):
        # Regex to detect common chapter headings
        chapters = re.split(r'(?i)Chapter\s+\d+|PART\s+[I|V|X]+', text)
        return [c.strip() for c in chapters if len(c.strip()) > 100]

    # Map the novel to a flattened table of chapters
    chapters_table = raw_novels.select(
        chunks=pw.apply(split_into_chapters, pw.this.data),
        filename=pw.this._metadata["path"]
    ).flatten(pw.this.chunks)

    # 4. Secondary Chunking: Recursive Token Splitting
    # For LLM compatibility, we further split large chapters into token-sized chunks.
    # TokenCountSplitter ensures we don't exceed model context limits.
    splitter = TokenCountSplitter(max_tokens=512)
    
    final_chunks = chapters_table.select(
        text=pw.this.chunks,
        doc_id=pw.this.filename,
        # Enrich with metadata for 'Evidence Rationale'
        word_count=pw.apply(lambda x: len(x.split()), pw.this.chunks)
    )

    return final_chunks

# Run the pipeline
chunks = get_preprocessing_pipeline("./data/Books/")
pw.debug.compute_and_print(chunks)

TypeError: TokenCountSplitter.__init__() got an unexpected keyword argument 'overlap'